<a href="https://colab.research.google.com/github/mdkamrulhasan/machine_learning_concepts/blob/master/notebooks/supervised/Baseline_RNN_solution_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Random Forest Model for Predicting `sm_surface`

This notebook demonstrates how to:
1. Load time-series training data (`training_data.csv`).
2. Create calendar features (year, month, week of year).
3. Create lag features from `sm_surface`.
4. Train a RNN regression model.
5. Generate predictions for future dates (**Oct 1, 2024 → Jan 01, 2025**).


In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta
import os
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import plotly.graph_objects as go
import pdb

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


# Final data (Test starts at Oct 1, 2025)
# base_path = '/content/drive/MyDrive/CIS 678 (Machine Learning)/moisture-data-publish-v1'
# Final data (Test starts at Oct 1, 2025)
base_path = 'drive/MyDrive/data/time-series/W26-Gaylord_MI-Test-start:2024-10-01/moisture-data-publish-v1'


Mounted at /content/drive


Settin up Parameters

In [ ]:
# Number of lags to consider
NB_LAGS = 10

target = 'sm_surface'
static_features = ['month']

APPLY_FEATURE_SCALING = True
INCORPORATE_STATIC_FEATURES = True


In [ ]:
lag_features = ['lag{}'.format(i) for i in range(NB_LAGS, 0, -1)]
features = static_features + lag_features

## Utility Functions

In [ ]:
def apply_feature_engineering(df, nb_lags=3, test_data=False):
    """Feature engineering"""

    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)

    if not test_data:
      # for i in range(1, nb_lags+1):
      for i in range(nb_lags, 0, -1):
        df['lag{}'.format(i)] = df['sm_surface'].shift(i)
    else:
      df['sm_surface'] = 1.0 # place holder
      # for i in range(1, nb_lags+1):
      for i in range(nb_lags, 0, -1):
        # Placeholder data (Not real)
        df['lag{}'.format(i)] = df_train.sm_surface.mean()

    return df.fillna(df.mean(numeric_only=True)).reset_index(drop=True)

In [ ]:
def get_rnn_features(df, incorporate_static_features=False):

  x_seq = df[lag_features].values
  x_seq = x_seq.reshape(x_seq.shape[0], x_seq.shape[1], 1)

  if not incorporate_static_features:
    return x_seq
  else:

    x_static = df[static_features].values
    x_static_expanded = np.repeat(x_static, NB_LAGS, axis=1)
    x_static_expanded = x_static_expanded.reshape(x_static_expanded.shape[0], x_static_expanded.shape[1], 1)
    x_combined = np.concat([x_seq, x_static_expanded], axis=-1)

    return x_combined


In [ ]:
def get_rolling_predictions_rnn(df_train, df_test):

  predictions = list(df_train.sm_surface.values[-NB_LAGS:])


  for index, row in df_test.iterrows():
    #print
    row_features = row[lag_features].values.reshape(1, -1)
    # updating the lag features (must be aligned with the lag definitions)
    row_features_lags_updated = list(row_features[0])[:-NB_LAGS] + predictions[-NB_LAGS:]

    test_arr = np.array(row_features_lags_updated).reshape(1, -1)
    test_arr = test_arr.reshape(test_arr.shape[0], test_arr.shape[1], 1) # rnn format adaptation

    #pdb.set_trace()

    #TO: bring in static feaures
    if INCORPORATE_STATIC_FEATURES:
      # TODO: use a loop if multiple features
      x_static = row[static_features].values.reshape(1, -1)
      x_static_expanded = np.repeat(x_static, NB_LAGS)
      x_static_expanded =  x_static_expanded.reshape(1, len(x_static_expanded), 1)
      # x_static_expanded = x_static_expanded.reshape(x_static_expanded.shape[0], x_static_expanded.shape[1], 1)

      test_arr = np.concat([test_arr, x_static_expanded], axis=-1)


    #pdb.set_trace()
    # pred = model.predict(test_arr)[0]
    # pred = model(test_arr)
    test_arr = np.array(test_arr, dtype=np.float32)
    X_test_tensor = torch.tensor(test_arr, dtype=torch.float32)
    y_pred = model(X_test_tensor).detach().numpy()[0][0]

    # pdb.set_trace()
    # _X = X.reshape(X.shape[0], X.shape[1], 1) # rnn format

    # updating the prediction tracking list
    predictions.append(y_pred)


  return predictions[NB_LAGS:]


## Load and Process Data

In [ ]:
# Training data
df_train = pd.read_csv(os.path.join(base_path, 'training_data.csv'))
df_train['date'] = pd.to_datetime(df_train['date'])
df_train = df_train.sort_values('date').reset_index(drop=True)
df_train = apply_feature_engineering(df_train, nb_lags=NB_LAGS)

### Load and Preprocess Test Data

In [ ]:
# Test data
df_test = pd.read_csv(os.path.join(base_path, 'test_data_X.csv'))
df_test['date'] = pd.to_datetime(df_test['date'])
df_test = df_test.sort_values('date').reset_index(drop=True)
df_test.head()

# Apply feature enginerring to test data
df_test = apply_feature_engineering(df_test, test_data=True, nb_lags=NB_LAGS)

In [ ]:
if APPLY_FEATURE_SCALING:
  scale_cols = [target] + features # scaling target to facilitate rolling predictions
  scaler = StandardScaler()
  # scaler model training
  df_train[scale_cols] = scaler.fit_transform(df_train[scale_cols])
  # scling test data
  df_test[scale_cols] = scaler.transform(df_test[scale_cols])

# RNN Modeling

### Data preprocessing

In [ ]:
X_train = get_rnn_features(df_train, incorporate_static_features=INCORPORATE_STATIC_FEATURES)
y_train = df_train[target].values

In [ ]:
# Define the RNN model
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNNModel, self).__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])
        return out


In [ ]:

# Create a custom dataset class for PyTorch DataLoader
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

### RNN Setup

In [ ]:
input_size = X_train.shape[2]
# Hyperparameters
hidden_size = 128
output_size = 1
learning_rate = 0.0001
num_epochs = 200
batch_size = 64

In [ ]:
# Create data loaders
train_dataset = TimeSeriesDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Initialize the model, loss function, and optimizer
model = RNNModel(input_size, hidden_size, output_size)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
print(model)

RNNModel(
  (rnn): LSTM(2, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


In [ ]:
# Training the model
for epoch in range(num_epochs):
    for inputs, targets in train_loader:
        # outputs = model(inputs)
        outputs = model(inputs).squeeze(output_size)
        loss = criterion(outputs, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [10/200], Loss: 0.7548
Epoch [20/200], Loss: 0.3685
Epoch [30/200], Loss: 0.4328
Epoch [40/200], Loss: 0.5341
Epoch [50/200], Loss: 0.4045
Epoch [60/200], Loss: 0.4554
Epoch [70/200], Loss: 0.4107
Epoch [80/200], Loss: 0.4086
Epoch [90/200], Loss: 0.3537
Epoch [100/200], Loss: 0.3032
Epoch [110/200], Loss: 0.2643
Epoch [120/200], Loss: 0.2393
Epoch [130/200], Loss: 0.5325
Epoch [140/200], Loss: 0.3416
Epoch [150/200], Loss: 0.3868
Epoch [160/200], Loss: 0.2783
Epoch [170/200], Loss: 0.2040
Epoch [180/200], Loss: 0.1982
Epoch [190/200], Loss: 0.1936
Epoch [200/200], Loss: 0.4890


### Validating on Training Data (No rolling predictions)
- Just to check if the training is wroking

In [ ]:
# Evaluation on the test set
model.eval()
with torch.no_grad():
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_pred_train = model(X_train_tensor).numpy()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=np.arange(len(y_train)),
    y=df_train['sm_surface'],
    mode='lines',
    name='Training Moisture Level Data'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(y_train)),
    y=y_pred_train.flatten(),
    mode='lines',
    name='Predited Moisture Level'
))

fig.update_layout(
    title='Ground Truth vs Predicted sm_surface',
    xaxis_title='Date',
    yaxis_title='sm_surface'
)

fig.show()


## Rolling Predictions (as no mositure data is available in the test data)

In [ ]:
predictions = get_rolling_predictions_rnn(df_train, df_test)

## Inverse Transformation of targets in the original scale

In [ ]:
# update test dataframe
df_test[target] = predictions
if APPLY_FEATURE_SCALING:
  df_train[scale_cols] = scaler.inverse_transform(df_train[scale_cols])
  df_test[scale_cols] = scaler.inverse_transform(df_test[scale_cols])

## Visualization

In [ ]:
trr_df = df_train[['date', 'sm_surface']].copy()
trr_df['is_train'] = True
tst_df = df_test[['date']].copy()
tst_df['sm_surface'] = df_test[target].values
tst_df['is_rain'] = False
combined_df = pd.concat([trr_df, tst_df], ignore_index=True)
combined_df['date'] = pd.to_datetime(combined_df['date'])
combined_df = combined_df.sort_values('date')


In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=combined_df[combined_df.is_train==True]['date'],
    y=combined_df[combined_df.is_train==True]['sm_surface'],
    mode='lines',
    name='Training Moisture Level Data'
))

fig.add_trace(go.Scatter(
    x=combined_df[combined_df.is_train==False]['date'],
    y=combined_df[combined_df.is_train==False]['sm_surface'],
    mode='lines',
    name='Predited Moisture Level'
))

fig.update_layout(
    title='Ground Truth vs Predicted sm_surface',
    xaxis_title='Date',
    yaxis_title='sm_surface'
)

fig.show()


## Exporting Predictions for Kaggle Challenge Upload

In [ ]:
#Pack results in a pandas dataframe
test_predictions = pd.DataFrame({
    'rowId': range(0, len(df_test)),
    'label': df_test[target].values})
# Exporting resluts
test_predictions.to_csv(os.path.join(base_path, 'sample_submission-rnn.csv'), index=False)